<a href="https://colab.research.google.com/github/alejandrocamargogarcia7-ui/predictive-maintenance-turbofan/blob/main/Inversiones_Bolsa_Millonaria_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pipeline Cuantitativo - Bolsa Millonaria Trii 2026
Este cuaderno contiene los scripts de extracción de datos, análisis técnico y visualización para monitorear la Bolsa de Valores de Colombia (BVC) mediante Python y `yfinance`.

## 1. Configuración del Entorno y Librerías
Asegúrate de instalar `yfinance` y `plotly` si ejecutas esto en un entorno local.

In [1]:
# Instalar de forma silenciosa (-q) las librerías necesarias para el pipeline financiero y visual
!pip install yfinance plotly pandas numpy -q

# 'yfinance' se utiliza para descargar datos históricos del mercado de Yahoo Finance
import yfinance as yf

# 'pandas' es la librería estándar para manipulación, estructuración y análisis de datos tabulares (DataFrames)
import pandas as pd

# 'numpy' permite realizar operaciones matemáticas optimizadas y trabajar con arreglos numéricos complejos
import numpy as np

# 'plotly.graph_objects' se usa para construir gráficos de alta personalización técnica (ej. velas japonesas, subgráficos)
import plotly.graph_objects as go

# 'plotly.express' ofrece una API de alto nivel para crear visualizaciones rápidas y elegantes (ej. gráficos de dispersión)
import plotly.express as px

# 'warnings' se usa para suprimir alertas de deprecación o mensajes de limpieza del sistema para mantener limpia la consola
import warnings
from plotly.subplots import make_subplots
warnings.filterwarnings('ignore')

## 2. Mega Radar Cuantitativo BVC
Escaneo del mercado completo para calcular RSI, MACD, Bandas de Bollinger y Volumen.

In [2]:
# Lista completa de tickers de empresas de la Bolsa de Valores de Colombia (BVC) indexados en Yahoo Finance con el sufijo '.CL'
bvc_tickers = [
    "ECOPETROL.CL", "PFBCOLOM.CL", "BCOLOMBIA.CL", "ISA.CL", "GEB.CL",
    "CELSIA.CL", "CORFICOLCF.CL", "PFAVAL.CL", "GRUPOSURA.CL", "PFSURA.CL",
    "BOGOTA.CL", "NUTRESA.CL", "PROMIGAS.CL", "MINEROS.CL", "TERPEL.CL",
    "ETB.CL", "ENKA.CL", "CNEC.CL", "CONCONCRET.CL", "PFGRUPOARG.CL",
    "GRUPOARGOS.CL", "BVC.CL", "ELCONDOR.CL", "FABRICATO.CL", "OCCIDENTE.CL",
    "VILLAS.CL", "PEI.CL", "GRUBOLIVAR.CL", "EXITO.CL", "BHI.CL", "ENEL.CL",
    "PAZRIO.CL", "PIZANO.CL", "TITAN.CL", "FAMILIA.CL", "HCOLSEL.CL", "ICOLCAP.CL"
]

print(f"Descargando datos para {len(bvc_tickers)} emisores de la BVC...")
# Descarga masiva del histórico diario de los últimos 6 meses para todos los tickers, agrupando las columnas por cada activo
data = yf.download(bvc_tickers, period="6mo", interval="1d", group_by="ticker", progress=False)

# Lista vacía donde acumularemos las métricas procesadas de cada acción individual
resultados = []

for ticker in bvc_tickers:
    try:
        # Extraemos el DataFrame correspondiente a un solo ticker de la descarga grupal
        df = data[ticker].copy()
        # Eliminamos filas que no tengan registro de precio de cierre ('Close') para evitar errores de cálculo
        df = df.dropna(subset=['Close'])

        # Si el activo no tiene suficiente historial (mínimo 50 días), lo descartamos para que los indicadores de 20 y 26 días sean fiables
        if df.empty or len(df) < 50:
            continue

        # --- CÁLCULO DE MEDIAS MÓVILES (SMA y EMA) ---
        # Media Móvil Simple (SMA) de 20 días: Identifica el soporte o resistencia de mediano plazo
        df['SMA_20'] = df['Close'].rolling(window=20).mean()
        # Media Móvil Exponencial (EMA) de 9 días: Reacciona de forma rápida al precio
        df['EMA_9'] = df['Close'].ewm(span=9, adjust=False).mean()
        # Media Móvil Exponencial (EMA) de 21 días: Funciona como un indicador de tendencia a corto plazo
        df['EMA_21'] = df['Close'].ewm(span=21, adjust=False).mean()

        # --- CÁLCULO DEL ÍNDICE DE FUERZA RELATIVA (RSI de 14 períodos) ---
        # Calculamos la diferencia del precio de cierre diario
        delta = df['Close'].diff()
        # Separamos las ganancias diarias (valores positivos)
        gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
        # Separamos las pérdidas diarias (valores negativos transformados a positivos)
        loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
        # Fuerza Relativa (RS)
        rs = gain / loss
        # El RSI oscila entre 0 y 100 indicando zonas extremas
        df['RSI_14'] = 100 - (100 / (1 + rs))

        # --- CÁLCULO DEL MACD (Moving Average Convergence Divergence) ---
        # Línea MACD: Diferencia entre la EMA rápida de 12 días y la EMA lenta de 26 días
        df['MACD_Line'] = df['Close'].ewm(span=12, adjust=False).mean() - df['Close'].ewm(span=26, adjust=False).mean()
        # Línea de Señal (Signal): EMA de 9 períodos aplicada sobre la Línea MACD
        df['MACD_Signal'] = df['MACD_Line'].ewm(span=9, adjust=False).mean()
        # Histograma MACD: La distancia entre la Línea MACD y su Señal
        df['MACD_Histogram'] = df['MACD_Line'] - df['MACD_Signal']

        # --- CÁLCULO DE LAS BANDAS DE BOLLINGER ---
        # Desviación estándar de los últimos 20 días
        std_20 = df['Close'].rolling(window=20).std()
        # Banda Superior: 2 desviaciones estándar por encima de la media de 20 días
        df['BB_Upper'] = df['SMA_20'] + (std_20 * 2)
        # Banda Inferior: 2 desviaciones estándar por debajo de la media de 20 días
        df['BB_Lower'] = df['SMA_20'] - (std_20 * 2)

        # --- CÁLCULO DEL ATR_14 (Average True Range) para medir volatilidad ---
        df['H-L'] = df['High'] - df['Low'] # Rango máximo de la sesión actual
        df['H-PC'] = abs(df['High'] - df['Close'].shift(1)) # Rango entre máximo de hoy y cierre de ayer
        df['L-PC'] = abs(df['Low'] - df['Close'].shift(1)) # Rango entre mínimo de hoy y cierre de ayer
        # El True Range (Rango Verdadero) es el valor máximo de estos tres escenarios de movimiento
        df['TR'] = df[['H-L', 'H-PC', 'L-PC']].max(axis=1)
        # El ATR es la media móvil de 14 días del True Range
        df['ATR_14'] = df['TR'].rolling(window=14).mean()
        # Volumen promedio de transacciones de los últimos 20 días
        df['Volumen_Prom_20d'] = df['Volume'].rolling(window=20).mean()

        # Extraemos la fila del último día registrado en la serie de datos histórica para el reporte final
        ultimo = df.iloc[-1]

        if pd.notna(ultimo['Close']) and pd.notna(ultimo['RSI_14']):
            # Calificamos la tendencia según el histograma MACD actual
            tendencia_macd = "ALCISTA" if ultimo['MACD_Histogram'] > 0 else "BAJISTA"
            # Clasificamos el estado del activo según los límites tradicionales de sobrecompra o sobreventa del RSI
            estado_rsi = "SOBRECOMPRA" if ultimo['RSI_14'] > 70 else ("SOBREVENTA" if ultimo['RSI_14'] < 30 else "NEUTRAL")

            # Construimos un diccionario con el resumen de métricas clave
            resultados.append({
                'Acción': ticker.replace('.CL', ''),
                'Precio ($)': round(ultimo['Close'], 0),
                'Volumen Prom': f"{ultimo['Volumen_Prom_20d']:,.0f}",
                'RSI': round(ultimo['RSI_14'], 1),
                'Señal RSI': estado_rsi,
                'MACD Hist': round(ultimo['MACD_Histogram'], 2),
                'Tendencia MACD': tendencia_macd,
                'Posición vs SMA20': "ARRIBA" if ultimo['Close'] > ultimo['SMA_20'] else "ABAJO",
                'Riesgo Diario (ATR%)': round((ultimo['ATR_14'] / ultimo['Close']) * 100, 2)
            })

    except Exception as e:
        # Si un ticker específico falla por delistado o falta de datos, la estructura 'try-except' previene que se caiga todo el script
        continue

# Convertimos la lista de resultados procesados en un DataFrame de Pandas para su visualización tabular ordenada
df_final = pd.DataFrame(resultados)
df_final = df_final[df_final['Volumen Prom'] != 'nan']

print("\n--- MEGA RADAR CUANTITATIVO BVC ---")
# Ordenamos de menor a mayor RSI para identificar primero las oportunidades en niveles de sobreventa (baratos)
print(df_final.sort_values(by='RSI', ascending=True).to_string(index=False))

Descargando datos para 37 emisores de la BVC...


ERROR:yfinance:
6 Failed downloads:
ERROR:yfinance:['FAMILIA.CL', 'BCOLOMBIA.CL', 'PIZANO.CL', 'PFBCOLOM.CL', 'ENEL.CL', 'PFSURA.CL']: YFPricesMissingError('possibly delisted; no price data found  (period=6mo) (Yahoo error = "No data found, symbol may be delisted")')



--- MEGA RADAR CUANTITATIVO BVC ---
    Acción  Precio ($) Volumen Prom  RSI   Señal RSI  MACD Hist Tendencia MACD Posición vs SMA20  Riesgo Diario (ATR%)
    PAZRIO         4.0      592,176  0.0  SOBREVENTA      -0.02        BAJISTA             ABAJO                  0.38
 OCCIDENTE     19100.0        2,116 11.6  SOBREVENTA     -72.16        BAJISTA             ABAJO                  0.62
    PFAVAL       785.0    2,401,482 19.7  SOBREVENTA      -4.98        BAJISTA             ABAJO                  1.70
CORFICOLCF     20820.0      281,952 34.4     NEUTRAL    -272.55        BAJISTA             ABAJO                  3.77
CONCONCRET       472.0      827,538 37.0     NEUTRAL      -4.29        BAJISTA             ABAJO                  3.18
   ICOLCAP     24983.0      338,061 37.1     NEUTRAL    -118.80        BAJISTA             ABAJO                  1.66
  PROMIGAS      6060.0      249,062 38.5     NEUTRAL       4.89        ALCISTA             ABAJO                  2.83
       ISA 

## 3. Radar Táctico Interactivo (Plotly)
Visualización de Velas Japonesas, Bandas de Bollinger y MACD para la entrada de precisión.

In [3]:
ticker = "ECOPETROL.CL"
print(f"Generando gráfica interactiva para {ticker}...")

# Descargamos los datos históricos diarios de los últimos 6 meses para este emisor individual
df = yf.download(ticker, period="6mo", interval="1d", progress=False)

# Si las columnas vienen en formato MultiIndex (debido a actualizaciones recientes en la API de yfinance), aplanamos los nombres de columna
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.droplevel(1)

# Aseguramos que las columnas se lean como Series unidimensionales planas utilizando '.squeeze()'
close_series = df['Close'].squeeze()
open_series = df['Open'].squeeze()
high_series = df['High'].squeeze()
low_series = df['Low'].squeeze()

# --- CÁLCULO DE LOS INDICADORES PARA LA GRÁFICA TÁCTICA ---
# Media Móvil Simple de 20 días
df['SMA_20'] = close_series.rolling(window=20).mean()
# Desviación estándar de 20 días
std_20 = close_series.rolling(window=20).std()
# Banda de Bollinger Superior
df['BB_Upper'] = df['SMA_20'] + (std_20 * 2)
# Banda de Bollinger Inferior
df['BB_Lower'] = df['SMA_20'] - (std_20 * 2)

# Línea MACD (EWM de 12 días - EWM de 26 días)
df['MACD_Line'] = close_series.ewm(span=12, adjust=False).mean() - close_series.ewm(span=26, adjust=False).mean()
# Línea Señal del MACD
df['MACD_Signal'] = df['MACD_Line'].ewm(span=9, adjust=False).mean()
# Histograma MACD para detectar la aceleración o desaceleración del momentum del mercado
df['MACD_Hist'] = df['MACD_Line'] - df['MACD_Signal']

# --- MAQUETADO DE LA FIGURA INTERACTIVA ---
# Diseñamos una estructura con 2 subgráficos compartiendo el mismo eje X para sincronizar los zooms
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    vertical_spacing=0.03, row_heights=[0.7, 0.3])

# Subgráfico 1: Velas Japonesas (Muestra la acción del precio diario)
fig.add_trace(go.Candlestick(x=df.index, open=open_series, high=high_series,
                             low=low_series, close=close_series, name='Precio'), row=1, col=1)

# Subgráfico 1: Bandas de Bollinger Superiores e Inferiores y Media de 20 días
fig.add_trace(go.Scatter(x=df.index, y=df['BB_Upper'], line=dict(color='rgba(173, 216, 230, 0.5)', width=1), name='BB Superior'), row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df['BB_Lower'], line=dict(color='rgba(173, 216, 230, 0.5)', width=1), fill='tonexty', name='BB Inferior'), row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df['SMA_20'], line=dict(color='orange', width=2), name='SMA 20'), row=1, col=1)

# Subgráfico 2: Histograma MACD codificado en verde (positivo/alcista) y rojo (negativo/bajista)
colores_macd = ['green' if val >= 0 else 'red' for val in df['MACD_Hist']]
fig.add_trace(go.Bar(x=df.index, y=df['MACD_Hist'], marker_color=colores_macd, name='Histograma MACD'), row=2, col=1)
# Subgráfico 2: Línea MACD y Línea Señal
fig.add_trace(go.Scatter(x=df.index, y=df['MACD_Line'], line=dict(color='blue', width=1.5), name='Línea MACD'), row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df['MACD_Signal'], line=dict(color='orange', width=1.5), name='Señal MACD'), row=2, col=1)

# Estilo final del lienzo: Título, ejes, plantilla oscura para trading, y ocultar el deslizador nativo inferior
fig.update_layout(title=f'Análisis Técnico de Alta Precisión: {ticker}',
                  yaxis_title='Precio (COP)', xaxis_rangeslider_visible=False,
                  template='plotly_dark', height=700)
# Renderizamos y desplegamos de manera interactiva la visualización en la notebook
fig.show()

Generando gráfica interactiva para ECOPETROL.CL...


```markdown
## 3.1. Selector de Análisis Táctico Interactivo
Utiliza el siguiente control para seleccionar de forma rápida cualquier emisor de la Bolsa de Valores de Colombia y visualizar su gráfico técnico avanzado.
```

In [4]:
# @title Selecciona un Activo para Análisis Detallado { run: "auto" }
# Lista de tickers disponibles limpia de sufijos para presentación
tickers_disponibles = sorted([t.replace('.CL', '') for t in bvc_tickers])

# Formulario interactivo de Google Colab
acción_seleccionada = "ETB" # @param ['BHI', 'BOGOTA', 'BVC', 'CELSIA', 'CNEC', 'CONCONCRET', 'CORFICOLCF', 'ECOPETROL', 'ELCONDOR', 'ENKA', 'ETB', 'EXITO', 'FABRICATO', 'GEB', 'GRUBOLIVAR', 'GRUPOARGOS', 'GRUPOSURA', 'HCOLSEL', 'ICOLCAP', 'ISA', 'MINEROS', 'NUTRESA', 'OCCIDENTE', 'PAZRIO', 'PEI', 'PFAVAL', 'PFGRUPOARG', 'PROMIGAS', 'TERPEL'] {type:"string"}

ticker_completo = f"{acción_seleccionada}.CL"
print(f"Generando análisis para {ticker_completo}...")

# Descarga del histórico de los últimos 6 meses
df_sel = yf.download(ticker_completo, period="6mo", interval="1d", progress=False)

if not df_sel.empty:
    if isinstance(df_sel.columns, pd.MultiIndex):
        df_sel.columns = df_sel.columns.droplevel(1)

    close_s = df_sel['Close'].squeeze()
    open_s = df_sel['Open'].squeeze()
    high_s = df_sel['High'].squeeze()
    low_s = df_sel['Low'].squeeze()

    # Cálculos técnicos
    df_sel['SMA_20'] = close_s.rolling(window=20).mean()
    std_20_sel = close_s.rolling(window=20).std()
    df_sel['BB_Upper'] = df_sel['SMA_20'] + (std_20_sel * 2)
    df_sel['BB_Lower'] = df_sel['SMA_20'] - (std_20_sel * 2)
    df_sel['MACD_Line'] = close_s.ewm(span=12, adjust=False).mean() - close_s.ewm(span=26, adjust=False).mean()
    df_sel['MACD_Signal'] = df_sel['MACD_Line'].ewm(span=9, adjust=False).mean()
    df_sel['MACD_Hist'] = df_sel['MACD_Line'] - df_sel['MACD_Signal']

    # Crear subgráficos
    fig_sel = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.03, row_heights=[0.7, 0.3])

    # Velas
    fig_sel.add_trace(go.Candlestick(x=df_sel.index, open=open_s, high=high_s, low=low_s, close=close_s, name='Precio'), row=1, col=1)
    fig_sel.add_trace(go.Scatter(x=df_sel.index, y=df_sel['BB_Upper'], line=dict(color='rgba(173, 216, 230, 0.4)', width=1), name='BB Superior'), row=1, col=1)
    fig_sel.add_trace(go.Scatter(x=df_sel.index, y=df_sel['BB_Lower'], line=dict(color='rgba(173, 216, 230, 0.4)', width=1), fill='tonexty', name='BB Inferior'), row=1, col=1)
    fig_sel.add_trace(go.Scatter(x=df_sel.index, y=df_sel['SMA_20'], line=dict(color='orange', width=2), name='SMA 20'), row=1, col=1)

    # MACD
    colores_hist = ['green' if val >= 0 else 'red' for val in df_sel['MACD_Hist']]
    fig_sel.add_trace(go.Bar(x=df_sel.index, y=df_sel['MACD_Hist'], marker_color=colores_hist, name='Histograma MACD'), row=2, col=1)
    fig_sel.add_trace(go.Scatter(x=df_sel.index, y=df_sel['MACD_Line'], line=dict(color='blue', width=1.5), name='Línea MACD'), row=2, col=1)
    fig_sel.add_trace(go.Scatter(x=df_sel.index, y=df_sel['MACD_Signal'], line=dict(color='orange', width=1.5), name='Señal MACD'), row=2, col=1)

    fig_sel.update_layout(title=f'Análisis Técnico de Precisión: {acción_seleccionada}', yaxis_title='Precio (COP)', xaxis_rangeslider_visible=False, template='plotly_dark', height=600)
    fig_sel.show()
else:
    print(f"No se encontraron datos de cotización recientes para {ticker_completo}.")

Generando análisis para ETB.CL...


## 4. Matriz de Riesgo y Momentum BVC
Análisis visual de dispersión para buscar activos con alta volatilidad para el concurso.

In [5]:
# Para maximizar la cobertura del análisis, utilizamos la lista completa de todas las acciones de la BVC (definidas en bvc_tickers)
print(f"Calculando Matriz de Volatilidad para las {len(bvc_tickers)} acciones de la BVC...")
# Descarga histórica del último mes con el fin de calcular el retorno mensual y la desviación intradiaria
data = yf.download(bvc_tickers, period="1mo", interval="1d", group_by="ticker", progress=False)

# Lista para consolidar los resultados de dispersión de riesgo-retorno
metricas = []
for ticker in bvc_tickers:
    try:
        # Limpiamos valores vacíos en la serie de precios de cierre
        df = data[ticker].dropna(subset=['Close'])
        # Se requieren al menos 15 registros de transacciones al mes para que la muestra sea estadísticamente relevante
        if len(df) < 15: continue

        # Retorno Porcentual del Mes: Porcentaje de ganancia o pérdida entre el primer y último día del mes observado
        retorno_mes = ((df['Close'].iloc[-1] / df['Close'].iloc[0]) - 1) * 100

        # Rango diario promedio como medida de volatilidad intradiaria: Diferencia máxima-mínima normalizada por el precio de cierre
        df['Rango_Diario'] = (df['High'] - df['Low']) / df['Close']
        volatilidad_promedio = df['Rango_Diario'].mean() * 100

        # Acumulamos las variables financieras de interés en la lista de métricas
        metricas.append({
            'Acción': ticker.replace('.CL', ''),
            'Retorno 1 Mes (%)': round(retorno_mes, 2),
            'Volatilidad Diaria (%)': round(volatilidad_promedio, 2)
        })
    except:
        # Manejo ágil de excepciones individuales para no interrumpir el análisis si una acción no cotizó en el mes
        continue

# Transformamos el set de métricas recopiladas en un DataFrame estructurado de Pandas
df_metricas = pd.DataFrame(metricas)

# --- GRAFICADO CON PLOTLY EXPRESS ---
# Creamos un gráfico de dispersión (Scatter Plot) donde:
# Eje X: Es la volatilidad diaria (Riesgo)
# Eje Y: Es el retorno acumulado a 1 mes (Rendimiento)
# Tamaño del punto: Representa el nivel de volatilidad
# Color del punto: Degradado basado en el rendimiento del activo (Verde=Ganancia, Rojo=Pérdida)
fig = px.scatter(df_metricas, x='Volatilidad Diaria (%)', y='Retorno 1 Mes (%)',
                 text='Acción', size='Volatilidad Diaria (%)',
                 color='Retorno 1 Mes (%)', color_continuous_scale='RdYlGn',
                 title='Matriz de Riesgo y Momentum BVC Completa (Último Mes)')

# Ajustamos el posicionamiento de los textos para que se visualicen limpios arriba de los puntos del gráfico
fig.update_traces(textposition='top center', marker=dict(line=dict(width=1, color='DarkSlateGrey')))
fig.update_layout(template='plotly_dark', height=600)
# Añadimos una línea horizontal de referencia en el punto de rendimiento cero (0%) para trazar una frontera entre ganadores y perdedores
fig.add_hline(y=0, line_dash="dash", line_color="white", opacity=0.5)

# Presentamos la matriz interactiva final
fig.show()

Calculando Matriz de Volatilidad para las 37 acciones de la BVC...


ERROR:yfinance:
6 Failed downloads:
ERROR:yfinance:['BCOLOMBIA.CL', 'FAMILIA.CL', 'PIZANO.CL', 'PFBCOLOM.CL', 'ENEL.CL', 'PFSURA.CL']: YFPricesMissingError('possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol may be delisted")')


```markdown
# Documentación Técnica Detallada: Pipeline Cuantitativo BVC

Este documento proporciona una explicación exhaustiva de la estructura,
lógica y matemática detrás del pipeline cuantitativo para la
**Bolsa de Valores de Colombia (BVC)**.

---

## 1. Arquitectura del Pipeline y Flujo de Datos

El pipeline está estructurado en cuatro fases secuenciales:

1. **Configuración e Importación:**
   * Instalación silenciosa de dependencias.
   * Preparación del entorno con librerías clave:
     `yfinance`, `pandas`, `numpy` y `plotly`.

2. **Mega Radar Cuantitativo BVC:**
   * Extracción masiva y paralela de 37 activos colombianos.
   * Ventana temporal: últimos 6 meses.
   * Cálculo de indicadores clave consolidando un DataFrame resumen.

3. **Radar Táctico Interactivo (Foco Individual y Selector):**
   * Descarga de datos para generar gráficos interactivos de velas.
   * Sincronización con un panel inferior del oscilador MACD.

4. **Matriz de Riesgo y Momentum:**
   * Análisis de dispersión a un mes.
   * Correlación de la volatilidad diaria promedio con el retorno mensual.

---

## 2. Explicación Matemática de los Indicadores

### A. Media Móvil Simple (SMA) de 20 Períodos
Representa el precio promedio de un activo durante 20 días.
Se utiliza para filtrar el "ruido" del mercado a corto plazo.
$$\text{SMA}_{20} = \frac{\sum_{i=1}^{20} \text{Precio}_i}{20}$$
En el código:
`df['SMA_20'] = df['Close'].rolling(window=20).mean()`

### B. Media Móvil Exponencial (EMA) de 9 y 21 Períodos
Asigna un mayor peso a los datos de precios más recientes.
Reacciona mucho más rápido a los cambios de tendencia.
$$\text{EMA}_{t} = \text{Precio}_{t} \times \left(\frac{2}{n+1}\right) + \text{EMA}_{t-1} \times \left(1 - \frac{2}{n+1}\right)$$
Donde $n$ es el número de períodos (9 o 21).

### C. Índice de Fuerza Relativa (RSI de 14 Períodos)
Oscilador de momentum que mide la velocidad del precio. Varía de 0 a 100.
* **Fuerza Relativa (RS):**
  $$RS = \frac{\text{Ganancia Promedio de 14 días}}{\text{Pérdida Promedio de 14 días}}$$
* **Fórmula del RSI:**
  $$RSI = 100 - \frac{100}{1 + RS}$$
* **Reglas del sistema:**
  * **RSI > 70:** Señal de **Sobrecompra** (posible sobrevaloración).
  * **RSI < 30:** Señal de **Sobreventa** (posible infravaloración).

### D. MACD (Moving Average Convergence Divergence)
Muestra la relación entre dos medias móviles del precio.
1. **Línea MACD:**
   $$\text{Línea MACD} = \text{EMA}_{12}(\text{Cierre}) - \text{EMA}_{26}(\text{Cierre})$$
2. **Línea de Señal (Signal):** EMA de 9 días de la Línea MACD.
   $$\text{Señal} = \text{EMA}_{9}(\text{Línea MACD})$$
3. **Histograma MACD:** Distancia entre la Línea MACD y su Señal.
   $$\text{Histograma} = \text{Línea MACD} - \text{Señal}$$
   * **Histograma > 0:** Tendencia y momentum alcista.
   * **Histograma < 0:** Tendencia y momentum bajista.

### E. Bandas de Bollinger
Indicador de volatilidad compuesto por tres curvas:
1. **Banda Media:** La SMA de 20 períodos.
2. **Banda Superior:** SMA de 20 períodos + 2 desviaciones estándar.
   $$\text{Banda Superior} = \text{SMA}_{20} + (2 \times \sigma_{20})$$
3. **Banda Inferior:** SMA de 20 períodos - 2 desviaciones estándar.
   $$\text{Banda Inferior} = \text{SMA}_{20} - (2 \times \sigma_{20})$$

### F. ATR (Average True Range) y Riesgo Diario
El ATR mide la volatilidad considerando los saltos de apertura (gaps).
1. **Rango Verdadero (True Range - TR):** Máximo entre:
   * $\text{Alto} - \text{Bajo}$
   * $|\text{Alto} - \text{Cierre Anterior}|$
   * $|\text{Bajo} - \text{Cierre Anterior}|$
2. **ATR_14:** Promedio móvil de 14 períodos de los TR.
3. **Riesgo Diario (ATR%):** Expresa el ATR como % del precio actual.
   Permite comparar la volatilidad de activos con precios nominales muy dispares.
   $$\text{Riesgo ATR\%} = \left( \frac{\text{ATR}_{14}}{\text{Precio Cierre}} \right) \times 100$$

---

## 3. Robustez de Código y Manejo de Errores

El script cuenta con técnicas para evitar fallos inesperados:

* **Bloques `try-except`:**
  Evitan que errores en un activo (como deslistados o sin datos)
  detengan la ejecución del radar completo.
* **Filtrado por Historial (`len(df) < 50`):**
  Evita calcular indicadores móviles con datos insuficientes.
* **Gestión de MultiIndex:**
  Aplana dinámicamente las columnas de Yahoo Finance
  utilizando `.droplevel(1)` y `.squeeze()`.
* **Filtrado por Volumen:**
  Elimina activos ilíquidos que distorsionan las estadísticas.

---

## 4. Interpretación de la Matriz de Riesgo y Momentum

El gráfico de la sección 4 ayuda a la toma de decisiones tácticas:

* **Eje Horizontal (Volatilidad Diaria):**
  Mide el rango de movimiento diario. A mayor % de volatilidad,
  mayor potencial de ganancia, pero con un nivel de riesgo superior.
* **Eje Vertical (Retorno 1 Mes):**
  Mide el desempeño neto del activo durante el último mes.
* **Frontera Cero (Línea Blanca):**
  Separa de forma visual los activos ganadores de los perdedores.
* **Escala de Colores (Rojo a Verde):**
  Identifica de un vistazo el momentum (Fuerza compradora/vendedora).
```